# 기존 M4 결과 비교 · H&M 완료 결과 확인 (학습 없음)

현재 GPU 학습은 그대로 두고 **별도 CPU 런타임**에서 실행하세요. GPU를 선택하거나 기존 학습 런타임을 변경하지 마세요.

- 기존 JSON과 체크포인트 파일 목록만 읽습니다. 학습·추가 평가·Drive 파일 저장/수정 없음.
- H&M 42·43·44는 **존재 확인 대상**입니다. 미완료 시드는 자동 실행하지 않습니다.
- original: `1 + 0.5*q_C*가격백분위*가격대적합`
- complementary: `1 + 0.5*q_C*(1-RBF_value_fit)`
- 두 M4의 적합도 정의도 다릅니다. 서로 다른 실행을 합친 평균이나 성공 판정은 만들지 않습니다.


## 1. Drive 연결 및 확인 코드 불러오기
코드는 고정 커밋에서 받아 임시 폴더에만 저장합니다. PyTorch와 학습 러너를 가져오지 않습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import tempfile, urllib.request, importlib.util

drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/논문/data')
REVISION = '3bff12b733179b29a860026d48e97baafc322862'
url = f'https://raw.githubusercontent.com/jung-un/clv-m2-lightgcn-runner/{REVISION}/clv_existing_m4_readout.py'
module_path = Path(tempfile.mkdtemp(prefix='m4-readonly-')) / 'clv_existing_m4_readout.py'
module_path.write_bytes(urllib.request.urlopen(url, timeout=60).read())
spec = importlib.util.spec_from_file_location('m4_readonly', module_path)
audit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)
print('학습 없음 | 원본 Drive 읽기 전용 | 코드:', REVISION)


## 2. 완료 결과 읽기
현재 실행 중인 arm은 완료 JSON이 생긴 이후에만 나타납니다. 미발견은 미실행·미완료·검색 범위 밖 경로 모두 가능하며, 실패를 뜻하지 않습니다.

In [ ]:
tables = audit.readout(DATA_ROOT)
def show(name, frame):
    print('\n' + name)
    print('발견된 행 없음' if frame.empty else frame.to_string(index=False))
show('H&M 시드별 완료 결과 존재 여부', tables['hm_presence'])
show('읽기 경고 / 중복 충돌 (있으면 판정 보류)', tables['warnings'])


## 3. Dunnhumby 원형·보완 M4 비교
각 실행의 같은 시드 M1을 기준으로 한 변화율입니다. 실행 경로를 유지하며, 설정·입력 동일성 확인 없이 원형−보완의 직접 우열은 판정하지 않습니다. seed 42·43·44 이외는 포함하지 않습니다.

In [ ]:
comparison = tables['within_run_comparison']
if not comparison.empty:
    m4 = comparison[(comparison.dataset == 'dunnhumby') & (comparison.role == 'M4 실제')]
    show('원형/보완 M4 — 각 실행의 M1 대비 전체 지표', m4)
else:
    print('짝지을 완료 결과가 없습니다. 학습은 실행하지 않습니다.')


## 4. 기존 M5의 결합 효과
같은 실행·시드 안에서 M5−M4 / M5−M2를 표시합니다. 여러 rho의 M2가 있어 짝이 모호하면 M5−M2를 생략합니다. 과거 gradient-isolated M2 결과는 이 읽기 범위에서 제외해 현재 개인이력 M2와 섞지 않습니다.

In [ ]:
if not comparison.empty:
    show('동일 실행의 M5 비교', comparison[comparison.role == 'M5'])


## 5. H&M 저장된 결과와 체크포인트
체크포인트는 파일 존재·크기만 확인합니다. 저장 epoch·optimizer·입력 해시 등은 읽지 않았으므로 **재사용 승인 목록이 아닙니다**.

In [ ]:
absolute = tables['absolute']
if not absolute.empty:
    show('H&M 완료 절대지표', absolute[absolute.dataset == 'hm'])
if not comparison.empty:
    show('H&M 동일 실행 내 비교', comparison[comparison.dataset == 'hm'])
checkpoints = tables['checkpoints']
if not checkpoints.empty:
    show('H&M 체크포인트 목록 (내용 미검증)', checkpoints[checkpoints.dataset == 'hm'])


## 6. 전체 원본 지표 및 출처 확인
아래 표까지 함께 확인해야 일부 지표만으로 결론 내리지 않을 수 있습니다. `revenue`는 기존 코드의 열 이름이며 **가격·구매금액 가중 적중값**입니다. 유의성·CLV 귀속·미래 가치 증가를 주장하지 않습니다.

In [ ]:
show('전체 절대지표 — 생략 없음', tables['absolute'])
show('실행별 설정·출처·작동 진단', tables['inventory'])
print('\n완료: 기존 결과 읽기만 수행했습니다. 추가 학습은 0회입니다.')
